In [132]:
%pip install matplotlib

Note: you may need to restart the kernel to use updated packages.


In [133]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt 
%matplotlib inline

In [134]:
#read all the words
words = open("C:/Users/Trader/Desktop/names.txt", 'r').read().splitlines()
words [:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [135]:
len(words)

32033

In [136]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}

In [137]:
# block size = context length aka how mnay tokesn used to predict. 
block_size = 3 
X, Y = [],[]

for w in words:
    context = [0] * block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context) # current running context
        Y.append(ix) # prediction for the above context
        # print (''.join(itos[i] for i in context), '===>', itos[ix]) # print the context and the Y prediction
        context = context[1:] + [ix] # crop and append

X = torch.tensor(X)
Y = torch.tensor(Y)

In [138]:
def build_dataset(words):
    # block size = context length aka how mnay tokesn used to predict. 
    block_size = 3 
    X, Y = [],[]
    
    for w in words:
        context = [0] * block_size
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context) # current running context
            Y.append(ix) # prediction for the above context
            # print (''.join(itos[i] for i in context), '===>', itos[ix]) # print the context and the Y prediction
            context = context[1:] + [ix] # crop and append
    
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    return X, Y
import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr,Ytr = build_dataset(words[:n1])
Xdev,Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])

In [139]:
len(Xtr)

182625

In [140]:
C = torch.randn((27,2))

In [141]:
# encoding an integer into the C matrix. 
F.one_hot(torch.tensor(5),num_classes=27).float() @ C 

tensor([-0.0213, -0.1321])

In [142]:
# so we are embedding the 27 characters (alphabet + . ) into a 2 dimensional space. 
# then we are making a vector of all of the 3 character sequences that we have ("emm") 
# think of it as rows = all the combos. 
# columns = 1 column for each letter. 
# then in each value in the table is a dot product representing a single character
emb = C[X]
emb.shape

torch.Size([228146, 3, 2])

In [143]:
# initializing weights and biases 
W1 = torch.randn(6,100)
b1 = torch.randn(100)
b1.shape

torch.Size([100])

In [144]:
# we want to do something like emb * W + b1 but because emb is not the same dimensions as C we cannot
# so this just rearranged the information from above to have now 12 rows with 6  values (all of the dot products for the 3 characters)
# just concatenating the 2nd dimension into 1 dimension. IE 
# torch.cat([emb[:,0,:], emb[:,1,:], emb[:,2,:]],1).shape 

In [145]:
# Simpler way to do the above concatenation is with UNBIND
# torch.unbind(emb,1)
# torch.cat(torch.unbind(emb,1),1)

In [146]:
# all of the above is just to say that there are various ways to 
#concatenate / manipulate the dimensionality of a tensor
# but the easiest way is just to use the view() function
# thus we can basically do what we wanted to do above.
# having python just infer what the first dimension should be
# (since the dimensions should be the same it can use W1 to get it)
h = emb.view(-1,6) @ W1 + b1

In [147]:
h.shape

torch.Size([228146, 100])

In [148]:
(emb.view(-1,6) @ W1).shape
b1.shape

torch.Size([100])

In [149]:
# initializing the weights and biases for the second layer. 
W2 = torch.randn((100,27))
b2 = torch.randn(27)

In [150]:
# calculating the logits. 
logits = h @ W2 + b2
logits.shape

torch.Size([228146, 27])

In [151]:
#exponetiating to get a probability
counts = logits.exp()

In [152]:
# normalizing so that the rows add up to 1
prob = counts / counts.sum(1,keepdims = True)
prob [1].sum()

tensor(1.)

In [153]:
# now getting the prob(Y) by indexing into the Y vector grabbing the correct value
# and then looking for it in the new Prob Matrix to get the probabilty. 
# Then with that probability find the - log likelihood loss which is --> 
# Taking the Log of Probs, summing across the array, then making it neg.
 # loss = -prob[torch.arange(12),Y].log().mean()

In [379]:
# OK all together now. 
Xtr.shape, Ytr.shape

(torch.Size([182625, 3]), torch.Size([182625]))

In [385]:
#generator to make it reproducable. 
g = torch.Generator().manual_seed(2147483647) 
# initialize all of the relevant tensors that we will need
C = torch.randn((27,10),generator = g)
W1 = torch.randn((30,300),generator = g)
b1 = torch.randn((300),generator = g)
W2 = torch.randn((300,27),generator = g)
b2 = torch.randn((27), generator = g)
# model parameters
parameters = [C, W1, b1, W2, b2]

for p in parameters:
    p.requires_grad = True
sum (p.nelement() for p in parameters)

17697

In [156]:
# This is explanation of the use of the new cross_entropy function from Pytorch. 
# below is just calculation cross entropy from before but we can do this more simply. 
#counts = logits.exp()
#prob = counts / counts.sum(1,keepdims = True)
#loss = -prob[torch.arange(12),Y].log().mean()

In [234]:
#lre = torch.linspace(-3,0,1000)
#lrs = 10 **lre

In [386]:
lri = []
lossi =  []
stepi = []
lr = 0.1              # starting learning rate
min_lr = 0.001      # never go below this
patience = 3000       # steps to wait without improvement
best_loss = float('inf')
last_improve = 0
smooth = None

In [389]:
for i in range (30000):
    
    #batching
    ix = torch.randint(0,Xtr.shape[0],(32,))
    
    # Forward Pass
    emb = C[Xtr[ix]]
    h = torch.tanh(emb.view(-1,30) @ W1 + b1) # 12,100
    logits = h @ W2 + b2 # 12, 27
    loss = F.cross_entropy(logits,Ytr[ix])
    #print (loss.item())
    
    #Backward Pass
    for p in parameters: 
        p.grad = None
    loss.backward()

    # learning rate (Halve learning rate every time we do not improve in 20 iterations.
    # smoothed loss: a running average, so one we are not as susceptible to volatility 
    l = loss.item()
    smooth = l if smooth is None else 0.99 * smooth + 0.01 * l

    # halve lr if no improvement in `patience` steps
    if smooth < best_loss:
        best_loss = smooth
        last_improve = i
    elif i - last_improve >= patience:
        lr = max(lr / 2, min_lr)
        last_improve = i   # restart the wait after halving
    
    # update 
    for p in parameters:
        p.data += -lr * p.grad
    
    #track stats
    #lri.append(lr)
    lossi.append(loss.item())
    stepi.append(i)

print (loss.item())        
print (lr)

2.3513758182525635
0.001


In [346]:
#plt.plot(lri,lossi)


In [388]:
# overall loss, not just the minibatch / training loss. 
emb = C[Xdev]
h = torch.tanh(emb.view(-1,30) @ W1 + b1) # 12,100
logits = h @ W2 + b2 # 12, 27
loss = F.cross_entropy(logits,Ydev)

print (loss.item())

2.321925640106201


In [163]:
# training split, dev / validation split, test split.
# 80%, 10%, 10%
# training the weights on training. 
# tuning hyperparameters with the validation. 
# final testing on the test. 
